# Mini-Project 4: Rainfall Pattern & Crop Suitability Analyser
**Scenario.** An agricultural extension officer wants to compare rainfall regimes across regions and advise farmers on which crops suit which months. Rainfall data are **illustrative** (mm per month, Jan–Dec).

Logic lives in `src/rainfall.py` (`Region`, `CropRule`, `RegionComparator`, similarity functions); tests are in `tests/test_rainfall.py`.

**Sources used in this notebook**
- FAO Land & Water, *Crop information: Maize* (500–800 mm per season for a medium-maturity grain crop) and *Crop information: Bean* (300–500 mm for a 60–120-day crop): https://www.fao.org/land-water/databases-and-software/crop-information
- FAO agro-ecological zones tables (growth cycles: lowland maize 110–130 days, *Phaseolus* bean 90–120 days): https://www.fao.org/4/t0742e/T0742E11.htm
- UCDA, *Uganda Coffee Profiles* (Robusta ≈ 1,200–1,450 mm/yr; Arabica highlands 920–1,650 mm/yr): https://ugandacoffee.go.ug ; Nakyagaba et al. (2024), *International Journal of Agronomy* (Robusta yields highest at 1,100–1,200 mm/yr).
- Kisembe et al. (2018), *Theoretical and Applied Climatology*, citing Basalirwa (1995): unimodal regime in the north, bimodal near the equator (main season Mar–May, shorter Sep–Nov): https://link.springer.com/article/10.1007/s00704-018-2643-x

In [ ]:
# --- Google Colab setup: does nothing when you run on your own computer ---
import os, subprocess, sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # change this after you push the repo to GitHub

if "google.colab" in sys.modules and not os.path.isdir("src"):
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError("Colab cannot find the repo's src/ folder. Either edit REPO_URL above, or upload and "
                           "unzip the repo first (see the README section 'Run in Google Colab').")
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)
print("Working folder:", os.getcwd())

In [ ]:
from dataclasses import replace
import math
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import distance

from src.rainfall import (
    MONTHS, CropRule, Region, RegionComparator, Status,
    cosine_similarity, default_crops, euclidean_distance, pearson_correlation,
    plot_monthly_boxplots, plot_rainfall_lines, plot_suitability_heatmap,
    simulate_years, suitability_reliability,
)

SEED = 42
Path("figures").mkdir(exist_ok=True)

kampala = Region("Kampala", [120, 140, 180, 200, 220, 180, 90, 70, 60, 100, 110, 130])
gulu    = Region("Gulu",    [8, 25, 75, 160, 190, 145, 170, 215, 175, 150, 60, 15])
mbarara = Region("Mbarara", [70, 85, 120, 140, 90, 25, 20, 55, 100, 125, 120, 90])
regions = [kampala, gulu, mbarara]
crops = default_crops()
regions

## Task 1: `Region`: descriptive statistics

In [ ]:
print(f"{'region':<9}{'annual mm':>10}{'mean mm/mo':>11}{'wettest':>14}{'driest':>14}{'CV':>7}")
for r in regions:
    w, d = r.wettest_month, r.driest_month
    print(f"{r.name:<9}{r.annual_total:10,.0f}{r.mean:11.1f}{w[0]+' '+format(w[1],'.0f'):>14}{d[0]+' '+format(d[1],'.0f'):>14}{r.cv:7.2f}")

CV here is the population st.dev ÷ mean of the 12 monthly values (the 12 months *are* the whole annual cycle, so ddof = 0). A higher CV means more strongly seasonal rainfall: Gulu (0.61) is the most seasonal, and its Jan–Feb are almost dry.

## Task 2: `CropRule` thresholds and classification of every month
Each crop has a suitable **monthly** band. The sources give *seasonal* (maize, beans) or *annual* (coffee) figures, so I convert them:

| crop | source figure | conversion | monthly band | consecutive good months needed |
|---|---|---|---|---|
| maize | 500–800 mm per season (FAO) | ÷ 4 months (110–130-day cycle) | 125–200 mm | 4 |
| beans | 300–500 mm per season (FAO) | ÷ 3.5 months (90–120-day cycle) | ≈ 86–143 mm | 3 |
| coffee | 1,100–1,650 mm per year (UCDA; Nakyagaba et al.) | ÷ 12 | ≈ 92–138 mm | 1 (perennial) |

Below the band → **drought risk**; above → **waterlogging risk**. **Caveat:** FAO figures are crop water *requirements*, so the lower bound (drought) is better supported than the upper bound, because whether surplus rain causes waterlogging also depends on soil, drainage and evapotranspiration, which this model ignores. A sensitivity check follows the table.

In [ ]:
for c in crops:
    print(f"{c.name:<7} good band {c.min_mm:6.1f} - {c.max_mm:6.1f} mm/month | needs {c.min_run} consecutive good month(s) | {c.source}")

labels = {r.name: r.monthly_labels(crops) for r in regions}
print()
print(f"{'month':<6}" + "".join(f"{r.name:<44}" for r in regions))
for i, m in enumerate(MONTHS):
    print(f"{m:<6}" + "".join(f"{labels[r.name][i]:<44}" for r in regions))

**Sensitivity of the thresholds.** Scale every band by 0.8 and 1.25 and count the good months per crop and region.

In [ ]:
print(f"{'region':<9}{'crop':<8}{'good months at x0.8 / x1.0 / x1.25':<38}planting windows at x1.0")
for r in regions:
    for c in crops:
        counts = [int((r.suitability(c.scaled(f)) == Status.GOOD).sum()) for f in (0.8, 1.0, 1.25)]
        wins = ", ".join(str(w) for w in r.planting_windows(c)) or "none"
        print(f"{r.name:<9}{c.name:<8}{str(counts):<38}{wins}")

The windows are **fragile**: scaling the bands by 20–25% can move a crop from no suitable months to several (Gulu beans: 0 → 5 months at ×1.25). The verdicts are therefore only as good as the thresholds, and the thresholds should be calibrated locally (NARO / UCDA trials) before real advice.

## Task 3: Cosine similarity done correctly
The earlier question used `math.cos()`. That function takes **one number (an angle in radians)** and returns its cosine; it knows nothing about two vectors. Cosine *similarity* is the cosine of the angle **between** two vectors: (a · b) / (‖a‖ ‖b‖).

In [ ]:
a, b = kampala.rainfall, gulu.rainfall
print("math.cos(a[0]) is just cos(120 rad):", round(math.cos(a[0]), 4), "-> says nothing about similarity\n")

mine = cosine_similarity(a, b)
scipy_val = 1 - distance.cosine(a, b)          # scipy returns the cosine DISTANCE = 1 - similarity
print(f"my cosine_similarity       = {mine:.10f}")
print(f"1 - scipy cosine distance  = {scipy_val:.10f}")
assert math.isclose(mine, scipy_val, rel_tol=1e-12)

for i in range(3):                              # verify every pair
    for j in range(3):
        assert math.isclose(cosine_similarity(regions[i].rainfall, regions[j].rainfall),
                            1 - distance.cosine(regions[i].rainfall, regions[j].rainfall), rel_tol=1e-12, abs_tol=1e-12)
print("All 9 region pairs agree with scipy.spatial.distance.cosine.")

## Task 4: Cosine similarity, Pearson correlation and Euclidean distance for every pair

In [ ]:
comp = RegionComparator(regions)
mats = comp.all_matrices()
for name, M in mats.items():
    print(f"\n{name}")
    print(f"{'':<9}" + "".join(f"{n:>10}" for n in comp.names))
    for n, row in zip(comp.names, M):
        print(f"{n:<9}" + "".join(f"{v:10.3f}" for v in row))

print("\nPearson check against np.corrcoef:", np.allclose(mats["Pearson correlation"], np.corrcoef([r.rainfall for r in regions])))
print("\nScale demo: Kampala vs a region with exactly 3x the rain every month")
wetter = 3 * kampala.rainfall
print(f"  cosine = {cosine_similarity(kampala.rainfall, wetter):.3f}, Pearson = {pearson_correlation(kampala.rainfall, wetter):.3f}, Euclidean = {euclidean_distance(kampala.rainfall, wetter):,.0f} mm")

**Why cosine similarity can call a much wetter region "similar".** It divides by the vector lengths, so it measures only the *direction* (shape) of the rainfall pattern and ignores the total amount: a region with exactly 3× Kampala's rain every month scores a perfect 1.000 even though it is 3,200 mm per year wetter (the Euclidean distance between the monthly vectors is 985 mm). Two further effects show up in the real matrices:
- All rainfall values are positive, so every pair of regions has a cosine of at least 0.75 here; the measure is compressed near 1 and discriminates poorly.
- Gulu and Mbarara score 0.75 on cosine but have a **negative** Pearson correlation (−0.17): their wet and dry months are partly out of phase. Pearson removes each region's average before comparing, so it is the better measure of *seasonal timing*; Euclidean distance is the measure of *amount*. Use cosine for shape, Pearson for timing, Euclidean for volume, and say which question is being asked.

## Task 5: Detecting rainy seasons with `scipy.signal.find_peaks`
Rainfall is cyclical (December borders January), so `Region.rainy_season_peaks` rotates the series to start at the driest month before calling `find_peaks`; a season peaking at the year boundary is then not missed. A peak counts as a separate rainy season only if its **prominence** is at least 25% of the annual range (max − min): it must stand clear of the troughs around it.

In [ ]:
expected = {"Kampala": "bimodal", "Gulu": "unimodal", "Mbarara": "bimodal"}
print(f"{'region':<9}{'peaks (25% rule)':<20}{'detected':<10}{'expected (Kisembe et al. 2018)':<32}match")
for r in regions:
    pk = [MONTHS[i] for i in r.rainy_season_peaks(0.25)]
    print(f"{r.name:<9}{', '.join(pk):<20}{r.modality(0.25):<10}{expected[r.name]:<32}{'yes' if r.modality(0.25) == expected[r.name] else 'NO'}")

print("\nPeaks found at different prominence thresholds (fraction of annual range):")
print(f"{'threshold':<11}" + "".join(f"{r.name:<22}" for r in regions))
for f in (0.05, 0.10, 0.20, 0.25, 0.30, 0.50):
    print(f"{f:<11}" + "".join(f"{('/'.join(MONTHS[i] for i in r.rainy_season_peaks(f))):<22}" for r in regions))
rng_k = kampala.rainfall.max() - kampala.rainfall.min(); rng_g = gulu.rainfall.max() - gulu.rainfall.min()
print(f"\nKampala's Dec bump has prominence 10 mm = {10/rng_k:.1%} of its range; Gulu's May peak has 45 mm = {45/rng_g:.1%} of its range.")

**Does it match Uganda's climate zones?** Partly.
- **Mbarara: bimodal** (peaks Apr and Oct) ✓. **Gulu: unimodal** ✓: its May and Aug peaks are separated by only a shallow June dip (145 mm against peaks of 190 and 215 mm), so they merge into one long season, in line with the unimodal northern regime (roughly Apr–Oct).
- **Kampala: unimodal ✗.** Kampala sits near the equator and is expected to be bimodal (main season Mar–May, shorter Sep–Nov). The illustrative Kampala series rises from September (60 mm) through December (130 mm) and into March–May without ever dipping, so there is no trough that separates a second season. It is a **property of the illustrative data, not necessarily of the method**; real Kampala records should be tested (see the extension).
- **No single prominence threshold fixes this.** Counting Kampala's weak December bump (needs ≤ ~6%) would also promote Gulu's May peak (22%) and any other wiggle, so Gulu would wrongly become bimodal. The sweep shows that the answer depends on the threshold, which is why it is an explicit, documented parameter.

## Task 6: Visualisation

In [ ]:
peaks = {r.name: r.rainy_season_peaks(0.25) for r in regions}
fig = plot_rainfall_lines(regions, peaks)
fig.savefig("figures/project4_rainfall_lines.png", dpi=150)
plt.show()

Mbarara shows two clear humps (Apr, Oct) separated by a deep Jun–Aug dry spell; Gulu's single hump spans Apr–Oct; Kampala has one dominant May peak with no clear second trough.

In [ ]:
fig = plot_suitability_heatmap(regions, crops)
fig.savefig("figures/project4_crop_heatmap.png", dpi=150)
plt.show()

Green cells are good for the crop; the numbers show the rainfall behind each verdict. Maize (4 consecutive months needed) finds a window only in Gulu; beans have the widest windows in Kampala (Oct–Feb) and Mbarara (Mar–May and Sep–Dec); Gulu's long wet season is too wet for beans and coffee under this FAO-derived upper bound (but see the sensitivity table above).

## Task 7: Advisory note (≤ 200 words), Mbarara

In [ ]:
NOTE = '''Advisory for Mbarara farmers (based on illustrative average rainfall).

Grow beans as your main food crop and treat maize as risky. Only April and October reach the 125-200 mm a month that maize needs, and maize needs four wet months in a row, so it is likely to suffer drought stress.

Beans fit Mbarara's two rainy seasons. Plant at the start of the second rains, in early September: rainfall then stays in the suitable range (about 86-143 mm a month) through December, which covers the 90-120 day bean cycle. A first-season planting in early March also works, but only March to May is suitable, which is just long enough, so plant as soon as the rains are established. The dry June and July help dry and harvest the first-season beans.

Be careful: these are average months. In some years rains start late or fail; in a simple variability simulation all four months of the September window were suitable in only about 6% of years. Plant early, stagger plantings by two weeks, and use mulch and drought-tolerant varieties. Coffee seedlings can go in with the September-November rains.'''
words = len(NOTE.split())
print(textwrap.fill(NOTE, 100, replace_whitespace=False))
print(f"\n[{words} words]")
assert words <= 200

## Extension: year-to-year variability (pipeline demonstrated on **synthetic** years)
The extension asks for ≥ 10 years of real rainfall (NASA POWER or UNMA). I had no internet access to download real series, so this cell shows the pipeline on **synthetic** data: 10 years drawn from a gamma distribution centred on Mbarara's monthly averages with an *assumed* CV of 0.35. **These are not observations.** To use real data, build a `(years, 12)` array of monthly totals in mm and pass it to the same functions.

In [ ]:
years10 = simulate_years(mbarara, n_years=10, cv=0.35, seed=SEED)
beans = crops[1]
fig = plot_monthly_boxplots(years10, "Mbarara: monthly rainfall over 10 SYNTHETIC years (assumed CV 0.35)", rule=beans)
fig.savefig("figures/project4_boxplots_synthetic.png", dpi=150)
plt.show()

years_big = simulate_years(mbarara, n_years=1_000, cv=0.35, seed=SEED)     # many years -> stable estimates
rel = suitability_reliability(years_big, beans)
print("Share of years in which each month is GOOD for beans (1,000 synthetic years):")
print("  " + "  ".join(f"{m} {v:.0%}" for m, v in zip(MONTHS, rel)))
good = beans.classify_array(years_big) == Status.GOOD
print(f"\nAll of Sep-Nov good: {good[:, 8:11].all(axis=1).mean():.1%} | all of Sep-Dec good: {good[:, 8:12].all(axis=1).mean():.1%} | all of Mar-May good: {good[:, 2:5].all(axis=1).mean():.1%}")

**Interpretation.** A month that is "good" on the *average* is good in only about half of the simulated years (Sep–Nov: 51–54%), and a whole four-month window is fully inside the band in only ~6% of years. So an average-based classification is optimistic, and the box plots show wide overlap between months, which is why the advisory recommends staggered planting rather than a single date. The 0.35 CV and the strict "every month inside the band" criterion are assumptions; real data would replace both, and could support a softer criterion such as total rainfall over the cycle.

## Findings & Limitations
**Findings.** Kampala is the wettest region (1,600 mm/yr), Mbarara the driest (1,040 mm) and Gulu the most seasonal (CV 0.61). Suitability depends on the rain pattern: maize finds a four-month window only in Gulu (Apr–Jul), while Mbarara's beans have two windows (Mar–May, Sep–Dec), matching its bimodal regime, so beans are the advice there. Cosine similarity (0.75–0.89) rates all regions alike because rainfall is non-negative and cosine ignores amount; Pearson (−0.17 to 0.21) shows their seasonal timing is largely unrelated, and Euclidean distance (250–315 mm) captures volume. Peak detection reproduces the published climate zones for Mbarara (bimodal) and Gulu (unimodal) but calls Kampala unimodal, contrary to the bimodal regime expected near the equator: the illustrative series never dries between its two rises, and no single prominence threshold fixes this without breaking Gulu.

**Limitations.** Thresholds come from seasonal crop *water needs* divided by cycle length; FAO gives need, not excess tolerance, so waterlogging flags are weaker than drought flags, and moving all bands by 20–25% changes windows dramatically (Gulu beans: 0 to 5 good months). Soil, temperature, altitude and drainage are ignored. Average months hide variability: in a synthetic simulation (assumed CV 0.35), a four-month window was fully suitable in only ~6% of years. All rainfall here is illustrative; real multi-year data are needed before any advice is given.